In [24]:
import pandas as pd

In [25]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [26]:
df["email_text"] = df["body"]

In [27]:
def derive_expected_action(triage_label):
    if triage_label == "respond":
        return "respond"
    elif triage_label == "ignore":
        return "ignore"
    elif triage_label == "notify_human":
        return "notify_human"
    else:
        return "unknown"


def derive_expected_tone(triage_label):
    if triage_label == "respond":
        return "polite"
    elif triage_label == "notify_human":
        return "neutral"
    else:
        return "neutral"

In [28]:
df["expected_action"] = df["triage_label"].apply(derive_expected_action)
df["expected_tone"] = df["triage_label"].apply(derive_expected_tone)

df[["triage_label", "expected_action", "expected_tone"]].head()

,triage_label,expected_action,expected_tone
0,notify_human,notify_human,neutral
1,respond,respond,polite
2,ignore,ignore,neutral
3,respond,respond,polite
4,respond,respond,polite


In [29]:
def email_assistant(email_text):
    text = email_text.lower()

    if "password" in text or "otp" in text:
        return {"action": "notify_human", "tone": "neutral"}
    elif "invoice" in text or "payment" in text:
        return {"action": "respond", "tone": "polite"}
    else:
        return {"action": "ignore", "tone": "neutral"}

In [30]:
def evaluate(prediction, expected_action, expected_tone):
    score = 0

    if prediction["action"] == expected_action:
        score += 1

    if prediction["tone"] == expected_tone:
        score += 1

    return score / 2  # max score = 1

In [31]:
scores = []

for _, row in df.iterrows():
    prediction = email_assistant(row["email_text"])
    
    score = evaluate(
        prediction,
        row["expected_action"],
        row["expected_tone"]
    )
    
    scores.append(score)

In [32]:
accuracy = (sum(scores) / len(scores)) * 100
accuracy

51.24999999999999

In [33]:
df["score"] = scores

output_path = "../data/milestone2_output.csv"
df.to_csv(output_path, index=False)

output_path

'../data/milestone2_output.csv'